# Tutorial: Generating Vector Embeddings

This notebook demonstrates how to generate vector embeddings for the sentences stored in SurrealDB using **Ollama**. These embeddings are essential for semantic search and AI-driven retrieval.

## Workflow:
1. **Fetch**: Retrieve sentences from the `sentence` table in SurrealDB.
2. **Embed**: Use Ollama to generate high-dimensional vectors for each sentence.
3. **Update**: Store the embeddings back into SurrealDB.

## Setup

We'll need the `requests` library to talk to Ollama and `surrealdb` to talk to our database.

In [ ]:
import os
import requests
from surrealdb import Surreal

# Configuration
SURREAL_URL = "ws://surrealdb:8000/rpc"
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
EMBED_MODEL = "mxbai-embed-large"

print(f"Using Ollama at: {OLLAMA_URL}")
print(f"Embedding Model: {EMBED_MODEL}")

## 1. Fetch Sentences from SurrealDB

We'll retrieve all sentences that don't have an embedding yet (or just all for this tutorial).

In [ ]:
def fetch_sentences():
    with Surreal(SURREAL_URL) as db:
        db.signin({"user": "root", "pass": "root"})
        db.use("main", "main")
        
        # Fetch sentences (Source: Quran)
        results = db.query("SELECT * FROM sentence WHERE source_type = 'quran'")
        return results[0]["result"]

sentences = fetch_sentences()
print(f"Fetched {len(sentences)} sentences.")

## 2. Generate Embeddings using Ollama

We'll define a helper function to call Ollama's embedding API.

In [ ]:
def get_embedding(text):
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={
            "model": EMBED_MODEL,
            "prompt": text
        }
    )
    return response.json()["embedding"]

# Test with the first sentence
sample_embedding = get_embedding(sentences[0]["text"])
print(f"Embedding Dimensions: {len(sample_embedding)}")

## 3. Update SurrealDB with Embeddings

Now we iterate through all sentences, generate their embeddings, and update the records.

In [ ]:
def update_embeddings(records):
    with Surreal(SURREAL_URL) as db:
        db.signin({"user": "root", "pass": "root"})
        db.use("main", "main")
        
        for record in records:
            print(f"Processing: {record['id']}...")
            embedding = get_embedding(record["text"])
            
            # Update the record with the embedding field
            db.query(
                "UPDATE $id SET embedding = $embedding",
                {"id": record["id"], "embedding": embedding}
            )

update_embeddings(sentences)
print("All embeddings updated successfully!")

## Conclusion

Your sentences now have vector embeddings! You can verify this by running a query in the SurrealDB dashboard or within this notebook. Next, you can implement semantic search using SurrealDB's vector indexing capabilities.